# Step 5: Fine-tune ReactionT5 on ORD (Model 1, Colab GPU)

Runs `scripts/train_reactant_model_ord.py`, which fine-tunes the ORD-pretrained
checkpoint `sagawa/ReactionT5v2-retrosynthesis` (before any USPTO-specific
fine-tuning) on the freshly built, leak-checked `data/v2_ord_train/reactants_train.jsonl`.

**Colab session budget: ~3h/day.** Checkpoints are written to Google Drive, and this
notebook can simply be re-run on a later day -- it auto-resumes from the last
checkpoint. Do not clear the Drive folder between sessions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

`data/v2_ord_train/` and `data/v2_uspto_train/` are gitignored (large, derived) --
regenerate them deterministically here (fixed seed, excludes the committed
`data/v2_ord_eval_targets.json`/`data/v2_uspto_eval_targets.json` by construction,
so this can never leak into the eval sets). Only needs to run once per Colab
session; skipped automatically if the files already exist (e.g. this is a
same-day resume).

In [ ]:
import os

if not os.path.exists("data/v2_ord_train/reactants_train.jsonl"):
    !python scripts/build_train_data_ord.py --pool-count 60000 --seed 42
if not os.path.exists("data/v2_uspto_train/reactants_train.jsonl"):
    !python scripts/build_train_data_uspto.py

**v2 result (for reference):** ORD-only data + lr=5e-5 + warmup + best-checkpoint
selection recovered from the earlier degraded run and gave a real improvement
over the pre-fine-tune baseline: exact_match 43.7%→50.7%, core_exact_match
54.3%→62.3% on the 300-target ORD eval set (USPTO also improved slightly,
16.3%→21.7%, with no USPTO data in training).

**v3 (this run): SMILES augmentation added.** The literature (Tetko et al.
2020; RSGPT, Nat. Commun. 2025) reports +10-14 points absolute top-1 accuracy
from training on randomized (non-canonical) SMILES instead of one fixed
canonical string per molecule -- it teaches the model the underlying
graph/chemistry invariance instead of memorizing a specific string encoding.
`train_reactant_model_ord.py` now does this online (a fresh random rendering
of product+reactants SMILES every epoch, train split only; validation stays
canonical for a stable eval_loss) -- on by default, no flag needed. Also
switched to Adafactor (smaller optimizer state, matters for Drive's 15GB
quota) and fixed a latent tied-embeddings save/reload edge case (verified
harmless via a local smoke run, but fixed for safety regardless).
`output_dir` below points at a new `model1_reactant_v3` folder.

In [ ]:
output_dir = "/content/drive/MyDrive/retro-planner-checkpoints/model1_reactant_v3"  # @param {type:"string"}
time_budget_minutes = 165  # @param {type:"number"}
mix_in_uspto = False  # @param {type:"boolean"}

extra_flag = ["--extra-train-file", "data/v2_uspto_train/reactants_train.jsonl"] if mix_in_uspto else []

In [ ]:
log_path = f"{output_dir}/train.log"

!python scripts/train_reactant_model_ord.py \
    --output-dir "{output_dir}" \
    --time-budget-minutes {time_budget_minutes} \
    {' '.join(extra_flag)} \
    > "{log_path}" 2>&1
print(f"Done (or paused at time budget). Log: {log_path}")

Training output is now redirected to `train.log` in `output_dir` (on Drive) instead of
printing in this cell -- a long run's per-step tqdm bar and log lines used to grow the
notebook's output DOM large enough to make the browser tab unresponsive after 1-2 hours,
even though the actual training was proceeding fine underneath. The cell above still
blocks until training stops (so Colab doesn't treat the runtime as idle), but prints
nothing while it runs. To check progress without waiting: open `train.log` directly in
Google Drive's own web preview (refresh it there) -- that works independently of the
Colab kernel, which stays busy running the cell above. The script was also changed to
log every 200 steps (was 25) and to skip the per-step progress bar entirely, so even the
log file itself stays compact.

Re-run the cell above (same `output_dir`) on the next day's Colab session to
continue training -- it auto-detects and resumes from the latest checkpoint.
Once `trainer.train()` finishes (not just time-budget-stopped), the final
model is also saved to `{output_dir}/final`.